# [P08 - Phase 2] Week 2: Recursive Bayesian State Estimation & Stochastic Baseline Policies
  
**Academic Reference Framework:** Moradi Afrapoli & Askari-Nasab (2019/2022); Hazrathosseini & Moradi Afrapoli (2024)

---

## 1. Mathematical Modeling & State Discretization

We model our system with a total state space of $3 \times 3 \times 2 = 18$ discrete operational scenarios. 

*   **State Indexing:** We map each state index $i \in \{0, \dots, 17\}$ to a structured tuple:
    $$\theta_i = (s, d, x)$$
    *   $s \in \{0 (\text{Low}), 1 (\text{Normal}), 2 (\text{High})\}$: Supply level.
    *   $d \in \{0 (\text{Low}), 1 (\text{Normal}), 2 (\text{High})\}$: Demand regime.
    *   $x \in \{0 (\text{None}), 1 (\text{Disrupted})\}$: Port disruption status.

We construct a transition matrix $T \in \mathbb{R}^{18 \times 18}$ representing state shifts, and a likelihood model $L(y_t \mid \theta_i)$ linking observed shipping delays and port queue size to the hidden states.

In [1]:
# WHAT: Initialize transition probabilities, likelihoods, and state mappings.
# INPUT: State discretization bounds.
# OUTPUT: Complete state definition dictionaries and probability matrices.

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

# Define state spaces
supply_states = ["Low", "Normal", "High"]      # Indices: 0, 1, 2
demand_states = ["Low", "Normal", "High"]      # Indices: 0, 1, 2
disrupt_states = ["No", "Yes"]                 # Indices: 0, 1

# Generate explicit mapping dictionary for 18 states
state_mapping = {}
idx = 0
for s_idx, s in enumerate(supply_states):
    for d_idx, d in enumerate(demand_states):
        for x_idx, x in enumerate(disrupt_states):
            state_mapping[idx] = {
                "supply": s_idx,     # 0, 1, 2
                "demand": d_idx,     # 0, 1, 2
                "disrupted": x_idx,  # 0, 1
                "label": f"S:{s}|D:{d}|X:{x}"
            }
            idx += 1

print(f"Discretization Schema: Successfully mapped {len(state_mapping)} joint latent states.")
print(f"Sample Mapping (State 7): {state_mapping[7]}")

Discretization Schema: Successfully mapped 18 joint latent states.
Sample Mapping (State 7): {'supply': 1, 'demand': 0, 'disrupted': 1, 'label': 'S:Normal|D:Low|X:Yes'}


## 2. Transition Tensor and Likelihood Model Construction

To govern transitions between states, we construct $T(j \mid i) = P(\theta_t = j \mid \theta_{t-1} = i)$. The transition tensor favors self-transitions (temporal consistency) while allowing stochastic jumps to adjacent states. 

The observation vector at time $t$, $y_t = (q_t, d_t)$, captures:
1.  **$q_t$ (Port Queue Length):** High queues indicate disruption $x=1$.
2.  **$d_t$ (Deficit / Shortfall kt):** Large delivery shortfalls point to low supply ($s=0$) or high demand ($d=2$).

In [3]:
# WHAT: Programmatically build the transition matrix T and observation likelihood estimator.
# INPUT: Predefined state mappings.
# OUTPUT: Standardized transition matrix T and likelihood evaluation function.

# Initialize standard transition matrix with high diagonal weights (system inertia)
T = np.zeros((18, 18))
for i in range(18):
    for j in range(18):
        # Calculate transition cost based on step differences
        s_diff = abs(state_mapping[i]["supply"] - state_mapping[j]["supply"])
        d_diff = abs(state_mapping[i]["demand"] - state_mapping[j]["demand"])
        x_diff = abs(state_mapping[i]["disrupted"] - state_mapping[j]["disrupted"])
        
        # Calculate exponential distance penalty
        distance_penalty = (s_diff * 1.5) + (d_diff * 1.5) + (x_diff * 2.0)
        T[i, j] = np.exp(-distance_penalty)
    
    # Normalize row to yield valid probabilities
    T[i, :] /= np.sum(T[i, :])

# Verify row stochasticity constraint
assert np.allclose(np.sum(T, axis=1), 1.0), "Transition matrix rows must sum to 1.0!"


def compute_likelihood(obs_queue, obs_shortfall, state_info):
    """
    Computes the conditional likelihood L(y_t | theta) of observing 
    a given queue length and delivery shortfall under state_info.
    """
    # 1. Queue Likelihood (modeled via Gamma distribution)
    # If disrupted (x=1), expect higher queues (mean=80, vs mean=15)
    if state_info["disrupted"] == 1:
        queue_likelihood = stats.gamma.pdf(obs_queue, a=4, scale=20)
    else:
        queue_likelihood = stats.gamma.pdf(obs_queue, a=2, scale=7.5)
        
    # 2. Shortfall Likelihood (modeled via Normal distribution)
    # High demand or low supply yields larger system shortfalls
    expected_shortfall = 0.0
    if state_info["supply"] == 0:  # Low supply
        expected_shortfall += 40.0
    if state_info["demand"] == 2:  # High demand
        expected_shortfall += 30.0
        
    shortfall_likelihood = stats.norm.pdf(obs_shortfall, loc=expected_shortfall, scale=15.0)
    
    # Joint likelihood under independence assumption
    return max(1e-9, queue_likelihood * shortfall_likelihood)

print("Transition matrix T and observation likelihood model successfully initialized.")

Transition matrix T and observation likelihood model successfully initialized.


## 3. Recursive Bayesian Belief Updating

Using our constructed matrices, we apply recursive Bayesian updating to track our state beliefs over time:

$$\bar{P}(\theta_t) = \sum_{\theta_{t-1}} T(\theta_t \mid \theta_{t-1}) P(\theta_{t-1} \mid y_{1:t-1})$$
$$P(\theta_t \mid y_{1:t}) = \frac{L(y_t \mid \theta_t) \bar{P}(\theta_t)}{\sum_{\theta_t'} L(y_t \mid \theta_t') \bar{P}(\theta_t')}$$

In [4]:
# WHAT: Recursive Bayesian state updates over sequential observations.
# INPUT: Sequential historical observations.
# OUTPUT: Evolution of posterior belief states over time.

def update_belief(prior_belief, obs_queue, obs_shortfall):
    """
    Performs one step of recursive Bayesian updating.
    
    Parameters:
        prior_belief (np.ndarray): Prior belief vector of shape (18,).
        obs_queue (float): Observed port queue size.
        obs_shortfall (float): Observed delivery shortfall.
        
    Returns:
        np.ndarray: Updated posterior belief vector of shape (18,).
    """
    # Step 1: Predict step using transition matrix (Markovian update)
    predicted_belief = np.dot(prior_belief, T)
    
    # Step 2: Update step incorporating observed likelihoods
    likelihoods = np.zeros(18)
    for idx in range(18):
        likelihoods[idx] = compute_likelihood(obs_queue, obs_shortfall, state_mapping[idx])
        
    posterior_unnormalized = predicted_belief * likelihoods
    
    # Normalize to yield a valid probability distribution
    posterior_sum = np.sum(posterior_unnormalized)
    if posterior_sum == 0.0:
        # Fallback to prior in case of numerical underflow
        return predicted_belief
        
    posterior_belief = posterior_unnormalized / posterior_sum
    return posterior_belief

# Perform quick test run
test_prior = np.full(18, 1.0 / 18.0)
updated_posterior = update_belief(test_prior, obs_queue=75.0, obs_shortfall=50.0)
print(f"Top 3 states in posterior distribution:")
sorted_states = np.argsort(updated_posterior)[::-1]
for i in range(3):
    state_id = sorted_states[i]
    print(f"  State {state_id:2d} ({state_mapping[state_id]['label']}): Prob = {updated_posterior[state_id]:.4f}")

Top 3 states in posterior distribution:
  State  3 (S:Low|D:Normal|X:Yes): Prob = 0.2893
  State  1 (S:Low|D:Low|X:Yes): Prob = 0.2718
  State 11 (S:Normal|D:High|X:Yes): Prob = 0.1485


## 4. Run Belief Tracker on Historical Logistics Observations

We test the accuracy of our Bayesian updates using simulated historical observations, tracking whether the recursive belief updates successfully detect simulated disruptions and market shifts.

In [ ]:
# WHAT: Historical simulation of observations to trace posterior belief paths.
# INPUT: Simulated monthly timeseries data.
# OUTPUT: Performance evaluation of state tracking accuracy.

# Generate synthetic historical monthly observations over 12 months
# Introduce a systemic port disruption (x=1) from Month 5 to Month 8
historical_data = pd.DataFrame({
    "Month": np.arange(1, 13),
    "Observed_Queue": [12.0, 15.0, 8.0, 14.0, 72.0, 85.0, 68.0, 78.0, 10.0, 16.0, 11.0, 15.0],
    "Observed_Shortfall": [5.0, 8.0, 2.0, 12.0, 45.0, 52.0, 38.0, 41.0, 4.0, 9.0, 6.0, 11.0],
    "True_Disruption": [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0] # Ground Truth for validation
})

# Sequential belief tracking loop
belief_history = []
current_belief = np.full(18, 1.0 / 18.0) # Start with uniform prior

for index, row in historical_data.iterrows():
    current_belief = update_belief(current_belief, row["Observed_Queue"], row["Observed_Shortfall"])
    belief_history.append(current_belief)

belief_history = np.array(belief_history)

# Extract marginalized probability of active disruption over time: P(X = 1)
prob_disrupted = np.zeros(12)
for t in range(12):
    for state_idx in range(18):
        if state_mapping[state_idx]["disrupted"] == 1:
            prob_disrupted[t] += belief_history[t, state_idx]

# Check tracking performance against ground truth
historical_data["Estimated_Disruption_Prob"] = prob_disrupted

# Output results
print("========== RECURSIVE BELIEF TRACKING ANALYSIS ==========")
for idx, r in historical_data.iterrows():
    print(f"Month {int(r['Month']):2d} | True Disrupted: {int(r['True_Disruption'])} | Estimated Disruption Prob: {r['Estimated_Disruption_Prob']*100:6.2f}%")
print("========================================================")

## 5. Comparative Evaluation: SAA Baseline vs. Myopic Baseline

We implement the two baseline policies to evaluate decision-making performance:

1.  **Baseline 1 - Sample Average Approximation (SAA):** Uses historical averages over 50 scenarios from Week 1 to optimize a static, single allocation policy for all future months.
2.  **Baseline 2 - Myopic (Greedy) Policy:** Allocates all available material dynamically to immediate, short-term demands and queue clearance.

In [5]:
# WHAT: Implementation of static SAA and dynamic Myopic allocations.
# INPUT: SimPy Monte Carlo scenario metrics.
# OUTPUT: Performance baseline KPIs.

# Load scenario metrics from Week 1 run
try:
    df_week1 = pd.read_csv('results/week1_monte_carlo_baseline.csv')
    avg_total_shipped = df_week1['Total_Shipped_kt'].mean()
    avg_total_backlog = df_week1['Total_Backlog_kt'].mean()
except FileNotFoundError:
    # Set fallback values if Week 1 dataset is missing
    avg_total_shipped = 1050.0
    avg_total_backlog = 85.0

# 1. SAA Baseline (Uses historical mean expectation as constant static decision rule)
saa_monthly_allocation = avg_total_shipped / 12.0
saa_expected_backlog = avg_total_backlog

# 2. Myopic Baseline (Greedy allocation that attempts to fully satisfy immediate demand)
# It handles fluctuations reactively without modeling future state transitions
myopic_allocations = []
myopic_backlogs = []

random_generator = np.random.default_seed(42)
for month in range(12):
    instantaneous_supply = random_generator.normal(100.0, 15.0)
    instantaneous_demand = random_generator.normal(90.0, 20.0)
    
    # Greedy allocation matching the minimum of supply and demand
    allocated = min(instantaneous_supply, instantaneous_demand)
    shortfall = max(0.0, instantaneous_demand - instantaneous_supply)
    
    myopic_allocations.append(allocated)
    myopic_backlogs.append(shortfall)

total_myopic_shipped = sum(myopic_allocations)
total_myopic_backlog = sum(myopic_backlogs)

print("========== BASELINE POLICIES PERFORMANCE ==========")
print("Baseline 1 - Static SAA Policy:")
print(f"  Mean Monthly Allocation Target : {saa_monthly_allocation:.2f} kt")
print(f"  Expected Horizon Backlog      : {saa_expected_backlog:.2f} kt")
print("-" * 51)
print("Baseline 2 - Reactive Myopic Policy:")
print(f"  Total Simulated Shipping      : {total_myopic_shipped:.2f} kt")
print(f"  Total Simulated Backlog       : {total_myopic_backlog:.2f} kt")
print("===================================================")

AttributeError: module 'numpy.random' has no attribute 'default_seed'

## 6. Descriptive Analytics: Diagnostic Performance Visualizations

We plot the marginalized belief trajectory against our true simulated disruption window to verify the accuracy of our recursive state updates.

In [6]:
# WHAT: Plot tracking diagnostics for state estimation accuracy.
# INPUT: Calculated historical estimates.
# OUTPUT: Performance visualization saved as 'results/belief_tracking_performance.png'.

plt.figure(figsize=(11, 5))
plt.plot(historical_data["Month"], historical_data["Estimated_Disruption_Prob"], 
         color='#d81b60', marker='o', linewidth=2.5, label='Posterior Disruption Belief P(X=1)')

plt.fill_between(historical_data["Month"], historical_data["True_Disruption"], 
                 color='#1e88e5', alpha=0.15, step="mid", label='True Disruption State')

plt.title('Recursive Bayesian Belief Update vs. True Port Disruption', fontsize=12, fontweight='bold')
plt.xlabel('Operational Month', fontsize=10)
plt.ylabel('Disruption Probability Profile', fontsize=10)
plt.xticks(historical_data["Month"])
plt.ylim(-0.05, 1.05)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='center left')
plt.tight_layout()

# Save performance chart
plt.savefig('results/belief_tracking_performance.png', dpi=300)
plt.show()

print("Belief tracking performance visualizations successfully exported!")

NameError: name 'historical_data' is not defined

<Figure size 1100x500 with 0 Axes>